## Importando as bibliotecas

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Carreagando e limpando os dados

In [12]:
sns.set_theme(style="whitegrid")

def load_and_clean_data(filepath):
    """Carrega e limpa os dados brutos do Investing.com"""
    df = pd.read_csv(filepath, parse_dates=[0], index_col=0, decimal=',', thousands='.') # Alteração: ajustando a colunda de data e colocando como index
    df.index = pd.to_datetime(df.index, format = "%d.%m.%Y")
    df = df.sort_index()

    def convert_volume(val):
        if isinstance(val, str):
            val = val.replace(',', '.')
            multipliers = {'B': 1e9, 'M': 1e6, 'K': 1e3}
            unit = val[-1]
            if unit in multipliers:
                return float(val[:-1]) * multipliers[unit]
        return 0

    df['Vol.'] = df['Vol.'].apply(convert_volume)
    return df

In [13]:
df =load_and_clean_data('/content/Dados Históricos - Ibovespa.csv')
#df = load_and_clean_data('/content/drive/MyDrive/dados/fase_02/Dados Históricos 02-03-24-03-03-26 - Ibovespa.csv')
df.head()

,Último,Abertura,Máxima,Mínima,Vol.,Var%
Data,,,,,,
2020-03-02,106625,104260,107220,103779,8860000.0,"2,35%"
2020-03-03,105537,106630,108804,104405,9390000.0,"-1,02%"
2020-03-04,107224,105540,107809,105042,8410000.0,"1,60%"
2020-03-05,102233,107217,107217,100536,8600000.0,"-4,65%"
2020-03-06,97997,102230,102230,96886,11860000.0,"-4,14%"


## Feature Engineering

In [14]:
def feature_engineering(df):
    """Cria indicadores técnicos e alvo temporal"""
    # Alvo: Fechamento de amanhã > hoje?
    df['Target'] = (df['Último'].shift(1) > df['Último']).astype(int) #Alterado o shifit(-1) para (1)

    # Indicadores
    df['Retorno_Diario'] = df['Último'] - df['Abertura'] # Alterado de pct_change para df['Último'] - df['Abertura']
    df['Range_Diario'] = (df['Máxima'] - df['Mínima']) / df['Abertura']
    df['Media_Movel_20'] = df['Último'].rolling(window=20).mean()
    df['Distancia_Media'] = df['Último'] - df['Media_Movel_20']
    df['Volatilidade_5d'] = df['Retorno_Diario'].rolling(window=5).std()

    # Lags
    for i in range(1, 4):
        df[f'Lag_Retorno_{i}'] = df['Retorno_Diario'].shift(i)
    return df.dropna()


# --- EXECUÇÃO ---
df_raw = load_and_clean_data('Dados Históricos - Ibovespa.csv')
df_final = feature_engineering(df_raw)


In [15]:
df_final.head()


,Último,Abertura,Máxima,Mínima,Vol.,Var%,Target,Retorno_Diario,Range_Diario,Media_Movel_20,Distancia_Media,Volatilidade_5d,Lag_Retorno_1,Lag_Retorno_2,Lag_Retorno_3
Data,,,,,,,,,,,,,,,
2020-03-27,73429,77708,77708,73057,10300000.0,"-5,51%",1,-4279,0.059852,82290.20,-8861.20,4872.437460,2754.0,5229.0,6125.0
2020-03-30,74639,73431,75430,73184,9030000.0,"1,65%",0,1208,0.030587,80690.90,-6051.90,4118.513482,-4279.0,2754.0,5229.0
2020-03-31,73020,74629,75511,72385,11140000.0,"-2,17%",1,-1609,0.041887,79065.05,-6045.05,3711.569116,1208.0,-4279.0,2754.0
2020-04-01,70967,73011,73011,69569,10090000.0,"-2,81%",1,-2044,0.047144,77252.20,-6285.20,2782.456559,-1609.0,1208.0,-4279.0
2020-04-02,72253,70969,73861,70957,10540000.0,"1,81%",0,1284,0.040919,75753.20,-3500.20,2359.335606,-2044.0,-1609.0,1208.0


## Definição de Features

In [16]:
features = ['Retorno_Diario', 'Range_Diario', 'Distancia_Media',
            'Volatilidade_5d', 'Lag_Retorno_1', 'Lag_Retorno_2', 'Vol.']
X = df_final[features]
y = df_final['Target']


## Separação de treino e teste

In [17]:
# Split Temporal (Respeitando a cronologia)
#split = int(len(df_final) * 0.8)
# Alterando para treinar com os últimos 30 dias
split = int(len(df_final.iloc[-30:]))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


## Treinando o modelo



In [18]:
# Modelo
model = RandomForestClassifier(n_estimators=200, max_depth=5,
                               random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)

## Realizando previsao

In [19]:
y_pred = model.predict(X_test)

## Acurácia

In [20]:
print(f"Acurácia Final: {accuracy_score(y_test, y_pred):.2%}")
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))

Acurácia Final: 89.00%

Relatório de Classificação:
               precision    recall  f1-score   support

           0       1.00      0.79      0.88       755
           1       0.81      1.00      0.90       691

    accuracy                           0.89      1446
   macro avg       0.91      0.89      0.89      1446
weighted avg       0.91      0.89      0.89      1446

